# VST / OmegaCAM pointing calculator

Give it the **target coordinates**; it returns the **pointing coordinates** to
enter in P2 and draws a schematic.

## Why an offset is needed

The OmegaCAM field centre falls exactly on the corner where chips 76/77/84/85
meet, i.e. inside the chip gap: the centre is only 0.7-7.5 arcsec from the
nearest chip edge, while the measured VST pointing error is **~25 arcsec** (and
its direction is not repeatable between runs).  Commanding the target
coordinates therefore usually puts the target on blank silicon.

Fix: place the target at the **centre of one of the four central chips**, which
leaves a 218 arcsec margin.

## Usage

Edit `RA_TARGET`, `DEC_TARGET`, `CHIP` in the second cell and run all cells.
The last cells verify the whole chain against the real 2026-11-16 observation.


In [ ]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle

plt.rcdefaults()
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
                     'mathtext.fontset': 'dejavusans',
                     'font.size': 23,
                     'axes.titlesize': 25,
                     'savefig.bbox': 'standard',
                     'savefig.format': 'png'})

# ---------------- detector geometry (measured from real VST/OmegaCAM data) ------
SCALE  = 0.21343                                            # arcsec / px
CHIP_W, CHIP_H = 2048, 4100                                 # px
XS = [0, 2138, 4280, 6422, 8562, 10701, 12844, 14984]       # column origins, W -> E
YS = [0, 4459, 8602, 13063]                                 # row origins, S -> N
FCX, FCY = 8527.0, 8562.0       # field centre (= commanded pointing), mosaic px


def chip_rect(n):
    """chip number -> (x0, y0, width, height) in mosaic pixels.
    chips 65-72 are the southern row, 89-96 the northern one."""
    row, col = divmod(n - 65, 8)
    return XS[col], YS[row], CHIP_W, CHIP_H


def chip_centre(n):
    x0, y0, w, h = chip_rect(n)
    return x0 + w / 2.0, y0 + h / 2.0


def offset_arcsec(x, y):
    """mosaic pixel -> (dEast, dNorth) arcsec relative to the field centre"""
    return (x - FCX) * SCALE, (y - FCY) * SCALE


def pointing_for(ra_t, dec_t, chip=77, pixel=None):
    """Target coordinates -> the RA/Dec to command.

    The target lands at the chosen detector position, which is offset from the
    field centre; so the boresight must sit at  target - offset.
    """
    t = SkyCoord(ra_t * u.deg, dec_t * u.deg)
    x, y = pixel if pixel is not None else chip_centre(chip)
    dE, dN = offset_arcsec(x, y)
    ang = np.arctan2(-dE, -dN) * u.rad
    return t.directional_offset_by(ang, np.hypot(dE, dN) * u.arcsec), (dE, dN)


def predict_pixel(ra_t, dec_t, cmd_ra, cmd_dec, pe=(0.0, 0.0)):
    """Where the target actually lands, given the commanded pointing and a
    pointing error pe = (dE, dN) in arcsec (actual - commanded)."""
    t = SkyCoord(ra_t * u.deg, dec_t * u.deg)
    c = SkyCoord(cmd_ra * u.deg, cmd_dec * u.deg)
    dE = (t.ra.deg - c.ra.deg) * 3600 * np.cos(c.dec.rad)
    dN = (t.dec.deg - c.dec.deg) * 3600
    return FCX + (dE - pe[0]) / SCALE, FCY + (dN - pe[1]) / SCALE


In [ ]:
# =============================== INPUT ===============================
RA_TARGET  = 345.81339        # deg, J2000
DEC_TARGET = -32.31699        # deg, J2000
CHIP       = 77               # 76 / 77 / 84 / 85 -- any of the four central chips
# PIXEL    = (9585.5, 6508.5)                 # or an explicit mosaic pixel
# =====================================================================

tgt = SkyCoord(RA_TARGET * u.deg, DEC_TARGET * u.deg)
cmd, (dE, dN) = pointing_for(RA_TARGET, DEC_TARGET, chip=CHIP)

print(f'target     RA, Dec = {tgt.ra.deg:10.5f}  {tgt.dec.deg:10.5f}   '
      f'({tgt.to_string("hmsdms", precision=2)})')
print(f'POINTING   RA, Dec = {cmd.ra.deg:10.5f}  {cmd.dec.deg:10.5f}   '
      f'({cmd.to_string("hmsdms", precision=2)})')
print()
d_ra = (cmd.ra.deg - tgt.ra.deg) * 3600 * np.cos(tgt.dec.rad)
d_dec = (cmd.dec.deg - tgt.dec.deg) * 3600
print(f'offset     dRA*cos(dec) = {d_ra:+7.1f}"    dDec = {d_dec:+7.1f}"')
print(f'           total = {np.hypot(d_ra, d_dec) / 60:.2f} arcmin')
print()
print(f'target lands at the centre of chip {CHIP}, '
      f'{CHIP_W / 2 * SCALE:.0f}" from the nearest chip edge')
print('observed VST pointing error ~25", direction not repeatable '
      '-> do NOT correct for it')


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(23, 10.5))

# ------------------------------------------------------- left: whole focal plane
ax = axs[0]
for n in range(65, 97):
    x0, y0, w, h = chip_rect(n)
    ax.add_patch(Rectangle((offset_arcsec(x0, y0)[0], offset_arcsec(x0, y0)[1]),
                           w * SCALE, h * SCALE, fc='0.93', ec='0.3', lw=0.8))
    ax.text(*offset_arcsec(x0 + w - 40, y0 + h - 40), n, ha='right', va='top',
            fontsize=19, color='C0')
ax.plot(0, 0, '+', color='C3', ms=26, mew=2.6)
ax.plot(dE, dN, 'o', mfc='none', mec='C2', mew=2.4, ms=17)
ax.annotate('', xy=(dE, dN), xytext=(0, 0),
            arrowprops=dict(arrowstyle='-|>', color='C2', lw=2.4))
ax.text(-45, 70, 'commanded pointing' + chr(10) + '(field centre, in the chip gap)',
        color='C3', fontsize=15, ha='right', va='bottom')
ax.text(dE + 45, dN, 'target' + chr(10) + f'(chip {CHIP} centre)', color='C2',
        fontsize=15, ha='left', va='center')
ax.set_xlim(-1850, 1850)
ax.set_ylim(-1850, 1850)
ax.set_aspect('equal')
ax.set_xlabel(r'$\Delta$R.A. $\times\cos\delta$  [arcsec]')
ax.set_ylabel(r'$\Delta$Dec  [arcsec]')
ax.set_title('whole focal plane')

# --------------------------------------------------------- right: chosen chip
ax = axs[1]
x0, y0, w, h = chip_rect(CHIP)
cx, cy = chip_centre(CHIP)
hw, hh = w * SCALE / 2, h * SCALE / 2
ax.add_patch(Rectangle((-hw, -hh), 2 * hw, 2 * hh, fc='0.93', ec='0.3', lw=1.4))
for rr, ls in ((60, ':'), (30, '--'), (10, '-')):
    ax.add_patch(plt.Circle((0, 0), rr, fill=False, ec='C2', lw=1.8, ls=ls, alpha=0.75))
ax.plot(0, 0, 'o', color='C2', ms=15)
ax.plot(-dE, -dN, '+', color='C3', ms=26, mew=2.6)
ax.annotate('', xy=(0, 0), xytext=(-dE, -dN),
            arrowprops=dict(arrowstyle='-|>', color='C3', lw=1.8, ls='--'))
ax.add_patch(plt.Circle((-dE, -dN), 25, fill=False, ec='C3', lw=1.6))
ax.text(50, -105, 'target' + chr(10) + '(nominal landing)', color='C2',
        fontsize=18, ha='left', va='top')
ax.text(-dE + 35, -dN - 60,
        'commanded pointing' + chr(10) + '(the RA, Dec you enter)' + chr(10)
        + 'circle: 25" pointing accuracy',
        color='C3', fontsize=17, ha='left', va='top')
ax.text(-hw + 30, -hh + 45,
        'circles: r = 10 / 30 / 60 arcsec' + chr(10) + 'possible pointing error',
        color='C2', fontsize=17, ha='left', va='bottom')
# default dither step shown as horizontal and vertical shifts
DX, DY = 25.0, 85.0
AX_, AY_ = 200.0, 480.0
ax.annotate('', xy=(AX_ + DX / 2, AY_), xytext=(AX_ - DX / 2, AY_),
            arrowprops=dict(arrowstyle='<|-|>', color='C1', lw=2.4))
ax.annotate('', xy=(AX_, AY_ + DY / 2), xytext=(AX_, AY_ - DY / 2),
            arrowprops=dict(arrowstyle='<|-|>', color='C1', lw=2.4))
ax.text(AX_ - DX / 2 - 12, AY_, '25"', color='C1', fontsize=15,
        ha='right', va='center')
ax.text(AX_ + 12, AY_ + DY / 2 - 10, '85"', color='C1', fontsize=15,
        ha='left', va='center')
ax.text(AX_, AY_ + DY / 2 + 28, 'default dither step', color='C1',
        fontsize=16, ha='center', va='bottom')
ax.plot([0, hw], [0, 0], color='0.5', lw=1.0, ls=':')
ax.text(hw / 2, 20, f'{hw:.0f}"', color='0.35', fontsize=18, ha='center')
ax.plot([0, 0], [0, hh], color='0.5', lw=1.0, ls=':')
ax.text(20, 170, f'{hh:.0f}"', color='0.35', fontsize=18)
ax.set_xlim(-hw - 350, hw + 140)
ax.set_ylim(-hh - 80, hh + 160)
ax.set_aspect('equal')
ax.set_xlabel(r'$\Delta$R.A. $\times\cos\delta$  [arcsec]')
ax.set_ylabel(r'$\Delta$Dec  [arcsec]')
ax.set_title(f'chip {CHIP} face-on')

plt.tight_layout()
plt.savefig(f'pointing_chip{CHIP}.png', dpi=120)
print(f'figure -> pointing_chip{CHIP}.png')
plt.show()


## Check against the real 2026-11-16 observation

For each frame: the pointing that was actually commanded, the measured pointing
error (actual - commanded, from `solve-field`), and the pixel where the target
was actually found in chip 77 (from the astrometric solution).  The next cell
predicts the pixel and compares.

In [ ]:
OBS = [
    # band, UT,        commanded RA,   commanded Dec,  dE,     dN,     x_meas, y_meas
    ('g', '00:34:57', 345.739250, -32.195220, 16.89, -22.57,  937.6, 2154.8),
    ('g', '00:40:43', 345.747476, -32.171620, 15.52, -22.60,  825.2, 1756.8),
    ('g', '00:46:28', 345.731020, -32.218830, 13.86, -21.97, 1070.1, 2550.0),
    ('r', '00:53:04', 345.739250, -32.195220, 13.26, -21.48,  955.2, 2149.1),
    ('r', '00:58:50', 345.747476, -32.171620, 11.73, -22.16,  843.4, 1754.1),
    ('r', '01:04:38', 345.731020, -32.218830, 10.96, -23.27, 1083.8, 2555.4),
]


def which_chip(x, y):
    for n in range(65, 97):
        x0, y0, w, h = chip_rect(n)
        if x0 <= x < x0 + w and y0 <= y < y0 + h:
            return n, x0, y0
    return None, x, y


print(f'{"frame":14s} {"chip":>5s} {"x_pred":>8s} {"x_meas":>8s} {"dx":>6s}'
      f' {"y_pred":>8s} {"y_meas":>8s} {"dy":>6s}   result')
for band, tm, cra, cdec, eE, eN, xm, ym in OBS:
    x, y = predict_pixel(RA_TARGET, DEC_TARGET, cra, cdec, (eE, eN))
    ch, cx0, cy0 = which_chip(x, y)
    xl, yl = x - cx0, y - cy0
    res = 'on chip %d' % ch if ch == CHIP else 'MISSED -> chip %s' % ch
    print(f'{band} {tm:11s} {str(ch):>5s} {xl:8.1f} {xm:8.1f} {xl - xm:+6.1f}'
          f' {yl:8.1f} {ym:8.1f} {yl - ym:+6.1f}   {res}')

print()
print('chip 77 spans x = 0..2047, y = 0..4099, so the target is well inside every frame;')
print('predicted and measured pixels agree to ~1-2 arcsec (a few px in x from the flat-sky approximation).')


## Notes

1. The `POINTING` line is what goes into P2; it differs from the target
   coordinates by about 8 arcmin.
2. The P2 finding chart will show the target about 8 arcmin from the field
   centre - that is correct, not an error.
3. **Do not** correct for the measured 25 arcsec pointing error: its direction
   flipped by 180 degrees between the two runs (2026-11-01/02: (-10.5", +18.9");
   2026-11-16: (+13.7", -22.3")), so it is not a constant.
4. Margins: 218 arcsec to the nearest chip edge, minus 85 arcsec of dither and
   25 arcsec of pointing error still leaves more than 100 arcsec.
5. To use another chip change `CHIP` (76/77/84/85); for an arbitrary position
   pass `PIXEL=(x, y)` in 0-based mosaic pixels.
